# 09. Atraso relativo ao prazo: baseline × regressão com features construídas

**Objetivo:** comparar um baseline com um modelo de regressão escolhido e configurado para este dataset, agora com **features construídas** a partir das colunas originais e dos joins entre as tabelas.

## Decisões

1. **Alvo simplificado.** Em vez de dias de atraso, o modelo prevê o **atraso relativo ao prazo ajustado**:

   `target_atraso_rel = max(0, duração real − prazo ajustado) / prazo ajustado`

   Isso tira o efeito do tamanho do contrato (um atraso de 30 dias pesa diferente em obras de 90 e de 900 dias). A duração real é `data_recebimento_provisorio − data_ordem_servico`, e o prazo ajustado inclui os aditivos de prazo. Para comparar com os notebooks anteriores, o resultado também é convertido em dias (`previsão relativa × prazo ajustado`). Como alternativa de leitura, o notebook mostra as faixas (0, 1 a 30 e mais de 30 dias) e mede o quanto a previsão separa os contratos que atrasam dos que não atrasam (AUC).
2. **Features construídas e sem redundância.** Todas partem de colunas originais, com razões e contagens que fazem sentido de negócio. Depois, uma poda remove as features muito correlacionadas entre si (|r| > 0,85), mantendo a mais simples.
3. **Dois cenários, para não misturar previsão com explicação:**
   - **Na ordem de serviço:** só informações conhecidas quando a obra começa.
   - **Com dados da execução:** acrescenta aditivos, parcelas de aditivo/supressão/reajuste e liberações, que só existem durante ou depois da obra.
4. **Modelo escolhido por validação temporal.** Candidatos: Ridge, Ridge com `log1p` no alvo, Random Forest raso e Gradient Boosting (perda Huber), cada um com uma grade pequena de hiperparâmetros. Para todos há também a opção de **limitar o alvo de treino em 1,0** (100% do prazo), que reduz o peso de atrasos extremos, como o do contrato 51 (380 dias de atraso em 60 de prazo). A avaliação continua usando o alvo real, sem limite.
5. **Critério de escolha: RMSE relativo.** Como 65% dos contratos têm atraso zero, o MAE sempre premia prever 0 (um modelo com perda L1 colapsa para a mediana). O RMSE penaliza não enxergar os atrasos grandes. O MAE, o erro em dias e a capacidade de ordenar os contratos (Spearman, AUC) são reportados ao lado, e o teste final só é usado uma vez, depois da escolha.
6. **Dois baselines constantes:** a mediana do treino (que é 0) e a média do alvo limitado, o equivalente constante dos modelos.

**Validação:** 4 janelas expanding (20→5, 25→5, 30→5, 35→5 contratos) nos 40 contratos mais antigos e teste final nos 8 mais recentes, ordenados por `data_recebimento_provisorio`.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

RANDOM_STATE = 42
N_TEST = 8
MIN_TRAIN = 20
BLOCK = 5
N_FOLDS = 4
LIMITE_CORRELACAO = 0.85

# Se os CSVs estiverem em outra pasta, defina DATA_DIR manualmente.
DATA_DIR = None

## 1. Carga dos dados

In [2]:
FILES = {
    "contratos": "vw_modelagem_contratos_filtros.csv",
    "aditivos_prazo": "vw_taditivoprazo.csv",
    "aditivos_valor": "vw_taditivovalor.csv",
    "liberacoes": "vw_tparcela_liberacao.csv",
    "parcelas": "vw_bi_parcelas.csv",
}


def find_data_dir():
    if DATA_DIR is not None:
        return Path(DATA_DIR)
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "data", cwd / "data" / "raw", Path("/mnt/user-data/uploads")]
    for parent in cwd.parents:
        candidates += [parent / "data", parent / "data" / "raw"]
    for folder in candidates:
        if (folder / FILES["contratos"]).exists():
            return folder
    raise FileNotFoundError("CSVs não encontrados; defina DATA_DIR.")


base = find_data_dir()


def read(name, **kw):
    return pd.read_csv(base / FILES[name], encoding="utf-8-sig", **kw)


contratos = read("contratos", parse_dates=["data_assinatura", "data_ordem_servico", "data_recebimento_provisorio"])
aditivos_prazo = read("aditivos_prazo", parse_dates=["Data_termino_anterior", "Data_termino_nova"])
aditivos_valor = read("aditivos_valor")
liberacoes = read("liberacoes")
parcelas = read("parcelas")

assert contratos["idContrato"].is_unique
ids = set(contratos["idContrato"])

# A seleção de contratos guia os joins: só entram linhas desses contratos.
aditivos_prazo = aditivos_prazo[aditivos_prazo["idContrato"].isin(ids)].copy()
aditivos_valor = aditivos_valor[aditivos_valor["idContrato"].isin(ids)].copy()
liberacoes = liberacoes[liberacoes["id_Contrato"].isin(ids)].copy()
parcelas = parcelas[parcelas["id_contrato"].isin(ids)].copy()

print("Pasta de dados:", base)
print({"contratos": len(contratos), "parcelas": len(parcelas), "liberacoes": len(liberacoes),
       "aditivos_prazo": len(aditivos_prazo), "aditivos_valor": len(aditivos_valor)})

Pasta de dados: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\raw
{'contratos': 48, 'parcelas': 897, 'liberacoes': 886, 'aditivos_prazo': 85, 'aditivos_valor': 134}


## 2. Alvo: atraso relativo ao prazo ajustado

Fim original = `data_ordem_servico + prazo_total`. Com aditivo, o fim ajustado é a maior `Data_termino_nova` do contrato (término vigente).

In [3]:
df = contratos.copy()

df["fim_original"] = df["data_ordem_servico"] + pd.to_timedelta(df["prazo_total"], unit="D")
fim_aditado = aditivos_prazo.groupby("idContrato")["Data_termino_nova"].max().rename("fim_aditado")
df = df.merge(fim_aditado, on="idContrato", how="left")

df["fim_ajustado"] = df[["fim_original", "fim_aditado"]].max(axis=1)
df["prazo_ajustado"] = (df["fim_ajustado"] - df["data_ordem_servico"]).dt.days
df["duracao_real"] = (df["data_recebimento_provisorio"] - df["data_ordem_servico"]).dt.days

df["target_dias_atraso"] = (df["duracao_real"] - df["prazo_ajustado"]).clip(lower=0)
df["target_atraso_rel"] = df["target_dias_atraso"] / df["prazo_ajustado"]
df["target_atraso"] = (df["target_dias_atraso"] > 0).astype(int)
df["faixa_atraso"] = pd.cut(df["target_dias_atraso"], [-1, 0, 30, np.inf], labels=["0 dia", "1 a 30 dias", "mais de 30 dias"])

assert (df["prazo_ajustado"] > 0).all()

display(df["faixa_atraso"].value_counts().sort_index().to_frame("contratos"))
display(df[["target_dias_atraso", "target_atraso_rel", "prazo_ajustado"]].describe().T)
display(
    df[df["target_atraso"] == 1][["idContrato", "prazo_total", "prazo_ajustado", "target_dias_atraso", "target_atraso_rel"]]
    .sort_values("target_atraso_rel", ascending=False)
    .head(8)
)

,contratos
faixa_atraso,
0 dia,31
1 a 30 dias,8
mais de 30 dias,9


,count,mean,std,min,25%,50%,75%,max
target_dias_atraso,48.000,31.104,73.405,0.000,0.000,0.000,25.750,380.000
target_atraso_rel,48.000,0.239,0.941,0.000,0.000,0.000,0.114,6.333
prazo_ajustado,48.000,311.812,262.816,60.000,131.250,200.500,397.000,"1,110.000"


,idContrato,prazo_total,prazo_ajustado,target_dias_atraso,target_atraso_rel
12,51,60,60,380,6.333
1,12,90,90,139,1.544
47,377,90,135,152,1.126
18,59,420,736,281,0.382
20,103,90,90,30,0.333
30,202,90,120,40,0.333
3,29,240,418,106,0.254
19,61,300,542,97,0.179


## 3. Features construídas (joins entre as tabelas)

Cada tabela auxiliar é ligada ao contrato pela chave (`id_contrato`, `id_Contrato` ou `idContrato`) e resumida por contrato; depois são calculadas razões que dependem do tamanho do contrato.

| Grupo | Features | Origem | Quando existe |
|---|---|---|---|
| Contrato | `prazo_total`, `log_valor_total`, `log_valor_por_dia`, `perc_desconto`, `dias_assinatura_ate_os`, `Classificacao_contrato` | `vw_modelagem_contratos_filtros` | ordem de serviço |
| Histórico | `fornecedor_contratos_anteriores`, `predio_contratos_anteriores` | contratos da seleção com OS anterior (mesmo fornecedor ou prédio) | ordem de serviço |
| Plano de parcelas | `n_parcelas_orig`, `prazo_por_parcela` | `vw_bi_parcelas` (tipo "Serviços Originais") | ordem de serviço |
| Aditivos de prazo | `n_aditivos_prazo`, `extensao_prazo_rel` (dias aditados / prazo original) | `vw_taditivoprazo` | durante a obra |
| Aditivos de valor | `n_aditivos_valor` | `vw_taditivovalor` | durante a obra |
| Escopo (parcelas) | `prop_parc_aditivo`, `prop_valor_aditivo`, `prop_parc_supressao`, `prop_parc_reajuste` (sobre as parcelas e o valor originais) | `vw_bi_parcelas` (tipos de aditivo, supressão e reajuste) | durante a obra |
| Liberações | `pct_lib_com_atraso`, `atraso_considerado_soma_rel`, `abono_atraso_soma_rel` (sobre o prazo original) | `vw_tparcela_liberacao` ligada às parcelas por `id_Contrato` + `id_Parcela` | durante e depois da obra |

Em `vw_bi_parcelas` o mesmo `id_parcela` se repete (execuções parciais, com os mesmos tipo, status e valor), então é usada uma linha por parcela.

In [4]:
d = df.sort_values(["data_ordem_servico", "idContrato"]).reset_index(drop=True)

# --- contrato
d["log_valor_total"] = np.log(d["valor_total_contrato"])
d["log_valor_por_dia"] = np.log(d["valor_total_contrato"] / d["prazo_total"])
d["perc_desconto"] = d["Perc_desconto"]
d["dias_assinatura_ate_os"] = (d["data_ordem_servico"] - d["data_assinatura"]).dt.days

# --- histórico (somente contratos com OS anterior)
d["fornecedor_contratos_anteriores"] = d.groupby("id_fornecedor").cumcount()
d["predio_contratos_anteriores"] = d.groupby("id_predio").cumcount()

# --- parcelas: uma linha por id_parcela
p = parcelas.drop_duplicates("id_parcela")
assert p.groupby("id_parcela")[["tipo_de_parcela", "valor_reajustado"]].nunique().max().max() == 1

TIPOS = {
    "orig": ["Serviços Originais"],
    "aditivo": ["Aditivo Mercado", "Aditivo Histórico"],
    "supressao": ["Supressão Histórico", "Supressão Mercado"],
    "reajuste": ["Reajuste"],
}

for nome, tipos in TIPOS.items():
    g = (
        p[p["tipo_de_parcela"].isin(tipos)]
        .groupby("id_contrato")
        .agg(**{f"n_{nome}": ("id_parcela", "nunique"), f"valor_{nome}": ("valor_reajustado", "sum")})
    )
    d = d.merge(g, left_on="idContrato", right_index=True, how="left")
    d[[f"n_{nome}", f"valor_{nome}"]] = d[[f"n_{nome}", f"valor_{nome}"]].fillna(0)

n_orig = d["n_orig"].replace(0, np.nan)
valor_orig = d["valor_orig"].replace(0, np.nan)

d["n_parcelas_orig"] = d["n_orig"]
d["prazo_por_parcela"] = d["prazo_total"] / n_orig
d["prop_parc_aditivo"] = d["n_aditivo"] / n_orig
d["prop_valor_aditivo"] = d["valor_aditivo"] / valor_orig
d["prop_parc_supressao"] = d["n_supressao"] / n_orig
d["prop_parc_reajuste"] = d["n_reajuste"] / n_orig

# --- aditivos
d = d.merge(aditivos_prazo.groupby("idContrato").size().rename("n_aditivos_prazo"), left_on="idContrato", right_index=True, how="left")
d = d.merge(aditivos_valor.groupby("idContrato").size().rename("n_aditivos_valor"), left_on="idContrato", right_index=True, how="left")
d[["n_aditivos_prazo", "n_aditivos_valor"]] = d[["n_aditivos_prazo", "n_aditivos_valor"]].fillna(0)
d["extensao_prazo_rel"] = (d["prazo_ajustado"] - d["prazo_total"]) / d["prazo_total"]

# --- liberações
lib = (
    liberacoes.assign(com_atraso=liberacoes["Atraso_considerado"] > 0)
    .groupby("id_Contrato")
    .agg(pct_lib_com_atraso=("com_atraso", "mean"), atraso_soma=("Atraso_considerado", "sum"), abono_soma=("Abono_atraso", "sum"))
)
d = d.merge(lib, left_on="idContrato", right_index=True, how="left")
d["atraso_considerado_soma_rel"] = d["atraso_soma"] / d["prazo_total"]
d["abono_atraso_soma_rel"] = d["abono_soma"] / d["prazo_total"]

# O join não pode duplicar nem perder contratos.
assert len(d) == len(contratos) == 48 and d["idContrato"].is_unique

In [5]:
# Candidatas em ordem de prioridade: na poda por correlação, a primeira da lista é a que fica.
CANDIDATAS_OS = [
    "prazo_total", "log_valor_total", "perc_desconto", "dias_assinatura_ate_os",
    "n_parcelas_orig", "prazo_por_parcela", "log_valor_por_dia",
    "fornecedor_contratos_anteriores", "predio_contratos_anteriores",
]
CANDIDATAS_EXEC = [
    "n_aditivos_prazo", "extensao_prazo_rel", "n_aditivos_valor",
    "prop_parc_aditivo", "prop_valor_aditivo", "prop_parc_supressao", "prop_parc_reajuste",
    "pct_lib_com_atraso", "atraso_considerado_soma_rel", "abono_atraso_soma_rel",
]
CATEGORICAS = ["Classificacao_contrato"]

todas = CANDIDATAS_OS + CANDIDATAS_EXEC
cobertura = pd.DataFrame({
    "faltantes": d[todas].isna().sum(),
    "zeros": (d[todas] == 0).sum(),
    "minimo": d[todas].min(),
    "maximo": d[todas].max(),
})
print("Cobertura das features candidatas (48 contratos):")
display(cobertura)
print("Contratos sem parcelas nem liberações (campos ficam vazios e são imputados pela mediana):",
      d.loc[d["n_orig"] == 0, "idContrato"].tolist())

Cobertura das features candidatas (48 contratos):


,faltantes,zeros,minimo,maximo
prazo_total,0,0,60.000,960.000
log_valor_total,0,0,11.218,16.801
perc_desconto,0,30,0.000,31.660
dias_assinatura_ate_os,0,0,9.000,267.000
n_parcelas_orig,0,1,0.000,32.000
prazo_por_parcela,1,0,11.538,60.000
log_valor_por_dia,0,0,6.503,10.261
fornecedor_contratos_anteriores,0,32,0.000,4.000
predio_contratos_anteriores,0,38,0.000,2.000
n_aditivos_prazo,0,11,0.000,7.000


Contratos sem parcelas nem liberações (campos ficam vazios e são imputados pela mediana): [12]


## 4. Separação desenvolvimento × teste final

In [6]:
ctr = d.sort_values(["data_recebimento_provisorio", "idContrato"]).reset_index(drop=True)

dev = ctr.iloc[:-N_TEST].copy()
test = ctr.iloc[-N_TEST:].copy()
assert dev["data_recebimento_provisorio"].max() <= test["data_recebimento_provisorio"].min()

folds = []
for k in range(N_FOLDS):
    n_train = MIN_TRAIN + k * BLOCK
    folds.append({"fold": k + 1, "train": dev.iloc[:n_train], "val": dev.iloc[n_train:n_train + BLOCK]})

print("Contratos de teste:", test["idContrato"].tolist())
display(pd.DataFrame([{
    "fold": f["fold"], "treino": len(f["train"]), "validacao": len(f["val"]),
    "com_atraso_validacao": int(f["val"]["target_atraso"].sum()),
    "atraso_rel_medio_validacao": f["val"]["target_atraso_rel"].mean(),
} for f in folds]))

Contratos de teste: [173, 201, 287, 283, 342, 317, 254, 377]


,fold,treino,validacao,com_atraso_validacao,atraso_rel_medio_validacao
0,1,20,5,3,0.130
1,2,25,5,2,0.099
2,3,30,5,0,0.000
3,4,35,5,2,0.035


## 5. Poda de features redundantes

Dentro de cada cenário, uma feature é descartada se tiver correlação absoluta acima de 0,85 com outra feature **já mantida** (a correlação é calculada só nos contratos de desenvolvimento e **não usa o alvo**).

In [7]:
def podar_redundantes(frame, candidatas, limite=LIMITE_CORRELACAO):
    corr = frame[candidatas].corr().abs()
    mantidas, descartadas = [], []

    for col in candidatas:
        if frame[col].nunique(dropna=True) <= 1:
            descartadas.append((col, "constante", None, None))
            continue
        parceiros = [m for m in mantidas if corr.loc[col, m] > limite]
        if parceiros:
            melhor = max(parceiros, key=lambda m: corr.loc[col, m])
            descartadas.append((col, "redundante", melhor, corr.loc[col, melhor]))
        else:
            mantidas.append(col)

    return mantidas, pd.DataFrame(descartadas, columns=["feature", "motivo", "parecida_com", "correlacao"])


CENARIOS = {}
for nome, candidatas in {
    "na ordem de serviço": CANDIDATAS_OS,
    "com dados da execução": CANDIDATAS_OS + CANDIDATAS_EXEC,
}.items():
    mantidas, descartadas = podar_redundantes(dev, candidatas)
    CENARIOS[nome] = mantidas
    print(f"== {nome}: {len(candidatas)} candidatas -> {len(mantidas)} mantidas")
    print("   mantidas:", mantidas)
    display(descartadas) if len(descartadas) else print("   nenhuma descartada")

== na ordem de serviço: 9 candidatas -> 7 mantidas
   mantidas: ['prazo_total', 'log_valor_total', 'perc_desconto', 'dias_assinatura_ate_os', 'prazo_por_parcela', 'fornecedor_contratos_anteriores', 'predio_contratos_anteriores']


,feature,motivo,parecida_com,correlacao
0,n_parcelas_orig,redundante,prazo_total,0.927
1,log_valor_por_dia,redundante,log_valor_total,0.894


== com dados da execução: 19 candidatas -> 17 mantidas
   mantidas: ['prazo_total', 'log_valor_total', 'perc_desconto', 'dias_assinatura_ate_os', 'prazo_por_parcela', 'fornecedor_contratos_anteriores', 'predio_contratos_anteriores', 'n_aditivos_prazo', 'extensao_prazo_rel', 'n_aditivos_valor', 'prop_parc_aditivo', 'prop_valor_aditivo', 'prop_parc_supressao', 'prop_parc_reajuste', 'pct_lib_com_atraso', 'atraso_considerado_soma_rel', 'abono_atraso_soma_rel']


,feature,motivo,parecida_com,correlacao
0,n_parcelas_orig,redundante,prazo_total,0.927
1,log_valor_por_dia,redundante,log_valor_total,0.894


## 6. Métricas, preparo e candidatos a modelo

In [8]:
def make_prep(num, cat):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
    ])


CAPS_ALVO = [None, 1.0]
CAP_ALVO_BASELINE = 1.0


def candidatos():
    lista = []
    for a in [10, 100, 1000, 10000]:
        lista.append(("Ridge", {"alpha": a}, lambda a=a: Ridge(alpha=a)))
        lista.append((
            "Ridge (log1p)", {"alpha": a},
            lambda a=a: TransformedTargetRegressor(Ridge(alpha=a), func=np.log1p, inverse_func=np.expm1),
        ))
    for depth in [2, 3]:
        for leaf in [3, 5]:
            lista.append((
                "Random Forest", {"max_depth": depth, "min_samples_leaf": leaf},
                lambda depth=depth, leaf=leaf: RandomForestRegressor(
                    n_estimators=300, max_depth=depth, min_samples_leaf=leaf, max_features=0.5,
                    random_state=RANDOM_STATE, n_jobs=-1),
            ))
    for depth in [1, 2]:
        for leaf in [3, 5]:
            lista.append((
                "Gradient Boosting (Huber)", {"max_depth": depth, "min_samples_leaf": leaf},
                lambda depth=depth, leaf=leaf: GradientBoostingRegressor(
                    loss="huber", n_estimators=100, learning_rate=0.05, max_depth=depth,
                    min_samples_leaf=leaf, subsample=0.8, random_state=RANDOM_STATE),
            ))
    return [(m, {**p, "cap_alvo": cap}, b, cap) for m, p, b in lista for cap in CAPS_ALVO]


def metricas(frame, col):
    pred_rel = frame[col].clip(lower=0)
    pred_dias = pred_rel * frame["prazo_ajustado"]
    real_dias = frame["target_dias_atraso"]

    duas_classes = frame["target_atraso"].nunique() == 2
    return {
        "mae_rel": mean_absolute_error(frame["target_atraso_rel"], pred_rel),
        "rmse_rel": np.sqrt(mean_squared_error(frame["target_atraso_rel"], pred_rel)),
        "mae_dias": mean_absolute_error(real_dias, pred_dias),
        "rmse_dias": np.sqrt(mean_squared_error(real_dias, pred_dias)),
        "spearman": spearmanr(frame["target_atraso_rel"], pred_rel)[0] if pred_rel.nunique() > 1 else np.nan,
        "auc_atrasa": roc_auc_score(frame["target_atraso"], pred_rel) if duas_classes else np.nan,
    }


def ajustar_prever(builder, num, cat, treino, alvo, cap=None):
    cols = num + cat
    y = treino["target_atraso_rel"]
    if cap is not None:
        y = y.clip(upper=cap)  # limita o peso de atrasos extremos só no treino
    modelo = Pipeline([("prep", make_prep(num, cat)), ("model", builder())])
    modelo.fit(treino[cols], y)
    return modelo, modelo.predict(alvo[cols])


def baseline_prever(treino, alvo, estrategia="median", cap=None):
    y = treino["target_atraso_rel"]
    if cap is not None:
        y = y.clip(upper=cap)
    b = DummyRegressor(strategy=estrategia).fit(treino[["prazo_total"]], y)
    return b.predict(alvo[["prazo_total"]])

## 7. Escolha do modelo por validação temporal

Para cada cenário, todos os candidatos são avaliados nas 4 janelas, com as previsões dos 20 contratos de validação agregadas, e ordenados pelo **RMSE relativo**. O teste final não entra nesta etapa.

> A escolha usa as mesmas janelas em que o desempenho é reportado, então o resultado da validação é otimista. A medida honesta é o teste final.

In [9]:
def cv_candidato(builder, num, cat, cap):
    oof = []
    for f in folds:
        _, pred = ajustar_prever(builder, num, cat, f["train"], f["val"], cap)
        oof.append(f["val"].assign(pred=pred, fold=f["fold"]))
    oof = pd.concat(oof)
    return pd.Series(metricas(oof, "pred")), oof


def cv_baseline(estrategia, cap):
    oof = pd.concat([
        f["val"].assign(pred=baseline_prever(f["train"], f["val"], estrategia, cap), fold=f["fold"]) for f in folds
    ])
    m = pd.Series(metricas(oof, "pred"))
    m[["spearman", "auc_atrasa"]] = np.nan  # previsão constante em cada janela: não ordena contratos
    return m, oof


BASELINES = {
    "baseline (mediana)": ("median", None),
    "baseline (média, alvo limitado)": ("mean", CAP_ALVO_BASELINE),
}
cv_baselines = {nome: cv_baseline(est, cap) for nome, (est, cap) in BASELINES.items()}

print("Baselines na validação (20 contratos agregados):")
display(pd.DataFrame({n: r[0] for n, r in cv_baselines.items()}).T)

SELECAO = "rmse_rel"
resultados = {}

for nome, num in CENARIOS.items():
    ranking, guardados = [], {}
    for modelo, params, builder, cap in candidatos():
        resumo, oof = cv_candidato(builder, num, CATEGORICAS, cap)
        ranking.append({"modelo": modelo, "parametros": str(params), **resumo.to_dict()})
        guardados[(modelo, str(params))] = (builder, cap, oof)

    ranking = pd.DataFrame(ranking).sort_values(SELECAO).reset_index(drop=True)
    melhor = ranking.iloc[0]
    builder, cap, oof = guardados[(melhor["modelo"], melhor["parametros"])]
    resultados[nome] = {"ranking": ranking, "melhor": melhor, "builder": builder, "cap": cap, "oof": oof}

    print(f"== Cenário: {nome} ({len(num)} features numéricas + {len(CATEGORICAS)} categórica) — ordenado por {SELECAO}")
    display(ranking.head(6))

Baselines na validação (20 contratos agregados):


,mae_rel,rmse_rel,mae_dias,rmse_dias,spearman,auc_atrasa
baseline (mediana),0.066,0.130,27.900,69.787,NaN,NaN
"baseline (média, alvo limitado)",0.119,0.132,36.130,57.297,NaN,NaN


== Cenário: na ordem de serviço (7 features numéricas + 1 categórica) — ordenado por rmse_rel


,modelo,parametros,mae_rel,rmse_rel,mae_dias,rmse_dias,spearman,auc_atrasa
0,Ridge (log1p),"{'alpha': 10000, 'cap_alvo': 1.0}",0.106,0.121,33.635,55.889,0.134,0.516
1,Ridge (log1p),"{'alpha': 1000, 'cap_alvo': 1.0}",0.106,0.122,33.701,55.865,0.011,0.451
2,Ridge (log1p),"{'alpha': 100, 'cap_alvo': 1.0}",0.114,0.130,35.412,58.299,-0.392,0.242
3,Ridge,"{'alpha': 10000, 'cap_alvo': 1.0}",0.118,0.132,36.078,57.242,0.173,0.538
4,Ridge,"{'alpha': 1000, 'cap_alvo': 1.0}",0.119,0.133,35.748,56.863,0.069,0.484
5,Ridge,"{'alpha': 100, 'cap_alvo': 1.0}",0.126,0.142,37.168,59.038,-0.349,0.264


== Cenário: com dados da execução (17 features numéricas + 1 categórica) — ordenado por rmse_rel


,modelo,parametros,mae_rel,rmse_rel,mae_dias,rmse_dias,spearman,auc_atrasa
0,Ridge (log1p),"{'alpha': 1000, 'cap_alvo': 1.0}",0.106,0.121,33.704,55.682,0.136,0.527
1,Ridge (log1p),"{'alpha': 10000, 'cap_alvo': 1.0}",0.106,0.121,33.635,55.860,0.221,0.571
2,Ridge (log1p),"{'alpha': 100, 'cap_alvo': 1.0}",0.109,0.128,35.326,60.953,-0.109,0.429
3,Gradient Boosting (Huber),"{'max_depth': 1, 'min_samples_leaf': 5, 'cap_alvo': None}",0.108,0.129,40.643,67.216,-0.332,0.253
4,Gradient Boosting (Huber),"{'max_depth': 1, 'min_samples_leaf': 5, 'cap_alvo': 1.0}",0.108,0.129,40.643,67.216,-0.332,0.253
5,Ridge,"{'alpha': 1000, 'cap_alvo': 1.0}",0.118,0.131,35.664,56.330,0.122,0.516


## 8. Teste final

In [10]:
linhas_final, tabelas_teste, importancias = [], {}, {}

for nome, num in CENARIOS.items():
    r = resultados[nome]
    melhor = r["melhor"]

    modelo_final, pred_modelo = ajustar_prever(r["builder"], num, CATEGORICAS, dev, test, r["cap"])

    t = test[["idContrato", "prazo_ajustado", "target_dias_atraso", "target_atraso_rel", "target_atraso"]].reset_index(drop=True)
    t["modelo"] = pred_modelo
    for rotulo, (est, cap) in BASELINES.items():
        t[rotulo] = baseline_prever(dev, test, est, cap)
    tabelas_teste[nome] = t

    descricao = f"{melhor['modelo']} {melhor['parametros']}"
    entradas = [(rotulo, rotulo, *cv_baselines[rotulo]) for rotulo in BASELINES]
    entradas.append((descricao, "modelo", melhor, r["oof"]))

    for rotulo, col, cv_res, _ in entradas:
        m_teste = metricas(t, col)
        if col != "modelo":  # previsão constante: não ordena contratos
            m_teste["spearman"] = m_teste["auc_atrasa"] = np.nan
        linhas_final.append({
            "cenario": nome, "modelo": rotulo,
            **{f"cv_{k}": cv_res[k] for k in ["mae_rel", "rmse_rel", "mae_dias", "rmse_dias", "spearman", "auc_atrasa"]},
            **{f"teste_{k}": v for k, v in m_teste.items()},
        })

    nomes = modelo_final["prep"].get_feature_names_out()
    est_final = modelo_final["model"]
    est_final = est_final.regressor_ if hasattr(est_final, "regressor_") else est_final
    valores = est_final.coef_ if hasattr(est_final, "coef_") else est_final.feature_importances_
    importancias[nome] = pd.Series(valores, index=nomes).sort_values(key=np.abs, ascending=False)

comparacao = pd.DataFrame(linhas_final)

print("Validação temporal (20 contratos agregados):")
display(comparacao[["cenario", "modelo", "cv_mae_rel", "cv_rmse_rel", "cv_mae_dias", "cv_rmse_dias", "cv_spearman", "cv_auc_atrasa"]])

print("Teste final (8 contratos):")
display(comparacao[["cenario", "modelo", "teste_mae_rel", "teste_rmse_rel", "teste_mae_dias", "teste_rmse_dias", "teste_spearman", "teste_auc_atrasa"]])

Validação temporal (20 contratos agregados):


,cenario,modelo,cv_mae_rel,cv_rmse_rel,cv_mae_dias,cv_rmse_dias,cv_spearman,cv_auc_atrasa
0,na ordem de serviço,baseline (mediana),0.066,0.130,27.900,69.787,NaN,NaN
1,na ordem de serviço,"baseline (média, alvo limitado)",0.119,0.132,36.130,57.297,NaN,NaN
2,na ordem de serviço,"Ridge (log1p) {'alpha': 10000, 'cap_alvo': 1.0}",0.106,0.121,33.635,55.889,0.134,0.516
3,com dados da execução,baseline (mediana),0.066,0.130,27.900,69.787,NaN,NaN
4,com dados da execução,"baseline (média, alvo limitado)",0.119,0.132,36.130,57.297,NaN,NaN
5,com dados da execução,"Ridge (log1p) {'alpha': 1000, 'cap_alvo': 1.0}",0.106,0.121,33.704,55.682,0.136,0.527


Teste final (8 contratos):


,cenario,modelo,teste_mae_rel,teste_rmse_rel,teste_mae_dias,teste_rmse_dias,teste_spearman,teste_auc_atrasa
0,na ordem de serviço,baseline (mediana),0.148,0.399,21.875,54.352,NaN,NaN
1,na ordem de serviço,"baseline (média, alvo limitado)",0.216,0.372,68.810,82.139,NaN,NaN
2,na ordem de serviço,"Ridge (log1p) {'alpha': 10000, 'cap_alvo': 1.0}",0.202,0.374,60.127,74.029,0.405,0.750
3,com dados da execução,baseline (mediana),0.148,0.399,21.875,54.352,NaN,NaN
4,com dados da execução,"baseline (média, alvo limitado)",0.216,0.372,68.810,82.139,NaN,NaN
5,com dados da execução,"Ridge (log1p) {'alpha': 1000, 'cap_alvo': 1.0}",0.195,0.363,57.019,70.576,0.327,0.667


In [11]:
for nome, t in tabelas_teste.items():
    print(f"== Teste final — {nome} (previsão em dias = previsão relativa × prazo ajustado)")
    out = t[["idContrato", "prazo_ajustado", "target_dias_atraso"]].rename(columns={"target_dias_atraso": "atraso_real_dias"})
    for col in ["baseline (mediana)", "baseline (média, alvo limitado)", "modelo"]:
        out[f"dias: {col}"] = t[col].clip(lower=0) * t["prazo_ajustado"]
    display(out)

== Teste final — na ordem de serviço (previsão em dias = previsão relativa × prazo ajustado)


,idContrato,prazo_ajustado,atraso_real_dias,dias: baseline (mediana),"dias: baseline (média, alvo limitado)",dias: modelo
0,173,1110,0,0.000,121.930,101.568
1,201,881,0,0.000,96.775,80.781
2,287,360,0,0.000,39.545,33.086
3,283,421,0,0.000,46.245,38.808
4,342,60,0,0.000,6.591,5.552
5,317,390,23,0.000,42.840,35.901
6,254,750,0,0.000,82.385,68.779
7,377,135,152,0.000,14.829,12.456


== Teste final — com dados da execução (previsão em dias = previsão relativa × prazo ajustado)


,idContrato,prazo_ajustado,atraso_real_dias,dias: baseline (mediana),"dias: baseline (média, alvo limitado)",dias: modelo
0,173,1110,0,0.000,121.930,90.344
1,201,881,0,0.000,96.775,82.669
2,287,360,0,0.000,39.545,31.189
3,283,421,0,0.000,46.245,37.278
4,342,60,0,0.000,6.591,5.760
5,317,390,23,0.000,42.840,33.288
6,254,750,0,0.000,82.385,63.084
7,377,135,152,0.000,14.829,16.461


## 9. O que o modelo escolhido usa

In [12]:
for nome, serie in importancias.items():
    print(f"== {nome} (coeficientes padronizados ou importâncias, conforme o modelo)")
    display(serie.head(8).to_frame("valor"))

== na ordem de serviço (coeficientes padronizados ou importâncias, conforme o modelo)


,valor
num__log_valor_total,-0.000
num__predio_contratos_anteriores,-0.000
num__dias_assinatura_ate_os,0.000
num__prazo_total,-0.000
num__prazo_por_parcela,-0.000
num__fornecedor_contratos_anteriores,-0.000
cat__Classificacao_contrato_Construção,-0.000
cat__Classificacao_contrato_Reforma Parcial,0.000


== com dados da execução (coeficientes padronizados ou importâncias, conforme o modelo)


,valor
num__pct_lib_com_atraso,0.002
num__atraso_considerado_soma_rel,0.002
num__log_valor_total,-0.002
num__extensao_prazo_rel,-0.002
num__abono_atraso_soma_rel,0.001
num__predio_contratos_anteriores,-0.001
num__prop_valor_aditivo,0.001
num__prop_parc_supressao,-0.001


## 10. Exportação do dataset de modelagem

In [13]:
OUTPUT_DIR = Path("09_dataset_modelagem_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

colunas_saida = (
    ["idContrato", "NumeroContrato", "data_ordem_servico", "data_recebimento_provisorio",
     "prazo_total", "prazo_ajustado", "target_dias_atraso", "target_atraso_rel", "target_atraso", "faixa_atraso"]
    + [c for c in CANDIDATAS_OS + CANDIDATAS_EXEC if c != "prazo_total"] + CATEGORICAS
)
ctr[colunas_saida].sort_values("idContrato").to_csv(
    OUTPUT_DIR / "dataset_modelagem_contratos.csv", index=False, encoding="utf-8-sig"
)
print("Salvo em:", (OUTPUT_DIR / "dataset_modelagem_contratos.csv").resolve(), "|", len(ctr), "contratos")

Salvo em: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\notebooks\09_dataset_modelagem_outputs\dataset_modelagem_contratos.csv | 48 contratos


## Leitura dos resultados

**O modelo escolhido melhora pouco o erro quadrático e vira, na prática, um preditor constante.** Nenhum modelo supera o baseline da mediana no erro médio absoluto.

| | MAE (dias) | RMSE (dias) | RMSE relativo |
|---|---|---|---|
| **Validação temporal (20 contratos)** | | | |
| Baseline (mediana = 0) | 27,9 | 69,8 | 0,130 |
| Baseline (média, alvo limitado) | 36,1 | 57,3 | 0,132 |
| Ridge `log1p`, cenário na ordem de serviço | 33,6 | 55,9 | 0,121 |
| Ridge `log1p`, cenário com dados da execução | 33,7 | 55,7 | 0,121 |
| **Teste final (8 contratos)** | | | |
| Baseline (mediana = 0) | 21,9 | 54,4 | 0,399 |
| Baseline (média, alvo limitado) | 68,8 | 82,1 | 0,372 |
| Ridge `log1p`, cenário na ordem de serviço | 60,1 | 74,0 | 0,374 |
| Ridge `log1p`, cenário com dados da execução | 57,0 | 70,6 | 0,363 |

- **O modelo escolhido ficou no extremo da regularização.** A busca preferiu `alpha` entre 1.000 e 10.000 (o limite da grade) e alvo limitado em 1,0 nos dois cenários. Os coeficientes ficam abaixo de 0,01 em módulo, então as features quase não influenciam a previsão. Ela fica em torno de 8% a 12% do prazo ajustado para todos os contratos.
- **Por isso o ganho sobre os baselines é pequeno.** Na validação, o RMSE relativo cai de 0,130 para 0,121, mas o MAE em dias piora (de 27,9 para 33,6): prever uma fração positiva do prazo troca erro absoluto por erro quadrático. A média com alvo limitado já faz quase o mesmo, e no teste fica praticamente igual ao modelo (0,372 contra 0,374).
- **A mediana segue melhor no MAE.** Como 31 dos 48 contratos têm atraso zero, prever 0 acerta a maioria. No teste, o MAE em dias é 21,9 para a mediana e cerca de 57 a 60 para o modelo.
- **Ordenação sem sinal consistente:** AUC de 0,52 e 0,53 na validação (chance é 0,5). No teste o AUC é 0,75 e 0,67, mas são 8 contratos e 2 atrasados, então o valor depende de poucos pares.
- **Dados da execução não mudaram a escolha.** O cenário com aditivos, parcelas de escopo e liberações selecionou a mesma família de modelo, com resultado quase idêntico ao cenário da ordem de serviço. Mesmo com informações que descrevem atraso por parcela, as features não generalizam entre as janelas temporais.
- **Poda de redundâncias:** `n_parcelas_orig` (correlação 0,93 com `prazo_total`) e `log_valor_por_dia` (0,89 com `log_valor_total`) foram descartadas. Ficaram 7 features numéricas na ordem de serviço e 17 com dados da execução.
- **Alvo relativo e atrasos extremos:** os contratos 51 (380 dias de atraso em 60 de prazo, 6,3 vezes o prazo) e 12 (1,5 vez) dominam o erro relativo. Limitar o alvo de treino em 1,0 apareceu em todas as melhores configurações.

**Ideias para testar depois (não executadas):** tratar como classificação de duas etapas (atrasa ou não, depois quanto), usar mais contratos (o filtro da seleção deixa de fora contratos que existem nas tabelas de aditivos), ou cortar os dados da obra em um ponto fixo do tempo para obter uma previsão utilizável durante a execução.